In [22]:
from pathlib import Path

from openai import AsyncOpenAI
from rich_struct import print_obj

from mini_rag.config import RetrieveConfig
from mini_rag.llm import OpenAIAnswerGenerator
from mini_rag.models import LlmAnswer
from mini_rag.prompts import build_context
from mini_rag.retrieve import retrieve_async

In [23]:
async def answer_question(question: str) -> LlmAnswer:
    config = RetrieveConfig(
        question=question,
        log_folder=Path("../logs"),
        index_dir=Path("../index"),
        chunks=Path("../index/chunks_600_90.jsonl"),
    )

    async with AsyncOpenAI() as client:
        search_results = await retrieve_async(config=config, client=client)

    question_context = build_context(search_results)

    async with AsyncOpenAI() as client:
        answer_provider = OpenAIAnswerGenerator(client, "gpt-5.4-mini")

        answer = await answer_provider.generate_async(question, question_context)

    return answer

In [24]:
answer = await answer_question("What is the purpose of IPCC Working Group III?")

22:24:51.422 INFO     Loading embeddings manifest from ../index/embedding_manifest.json
22:24:51.431 INFO     Loading normalized embeddings from ../index/embeddings.npy
22:24:51.434 INFO     Reading chunks from ../index/chunks_600_90.jsonl
22:24:51.666 DEBUG    Searching for text What is the purpose of IPCC Working Group III?
22:24:51.667 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
22:24:52.213 DEBUG    OpenAI returned 1 embeddings (total_tokens: 10) [Batch: 1]
22:24:52.215 DEBUG    Semantic searching
22:24:52.224 INFO     Quering OpenAI's model 'gpt-5.4-mini'
22:24:55.839 INFO     OpenAI model 'gpt-5.4-mini' responded in 3.61 seconds (input_tokens: 3099, output_tokens: 45, total_tokens: 3144) [Response: resp_046103a0f07c0994006afda7253a1487d0b23110e8c8d88cbc]

In [25]:
print_obj(answer)

LlmAnswer(                                                                                                         
    answer='IPCC Working Group III assesses mitigation response options, meaning options for reducing climate      
change [1].',                                                                                                      
    cited_refs=[                                                                                                   
        1                                                                                                          
    ],                                                                                                             
    insufficient_evidence=False                                                                                    
)                                                                                                                  

In [26]:
answer2 = await answer_question("How much did Apple earn in fiscal year 2025?")

22:25:59.026 INFO     Loading embeddings manifest from ../index/embedding_manifest.json
22:25:59.035 INFO     Loading normalized embeddings from ../index/embeddings.npy
22:25:59.038 INFO     Reading chunks from ../index/chunks_600_90.jsonl
22:25:59.322 DEBUG    Searching for text How much did Apple earn in fiscal year 2025?
22:25:59.323 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
22:26:00.188 DEBUG    OpenAI returned 1 embeddings (total_tokens: 12) [Batch: 1]
22:26:00.189 DEBUG    Semantic searching
22:26:00.201 INFO     Quering OpenAI's model 'gpt-5.4-mini'
22:26:03.386 INFO     OpenAI model 'gpt-5.4-mini' responded in 3.18 seconds (input_tokens: 3145, output_tokens: 38, total_tokens: 3183) [Response: resp_0c144503719160c0006afda7693abc87d0978de8d392b40c0d]

In [28]:
print_obj(answer2)

LlmAnswer(                                                                                                         
    answer="I'm sorry but based on provided document fragments I can't answer the question",                       
    cited_refs=[],                                                                                                 
    insufficient_evidence=True                                                                                     
)                                                                                                                  